# FLIR flash-method thermal analysis in Google Colab

Use this notebook to run the repository workflow directly from GitHub in Colab. It installs the repository requirements, lets you point to videos in Google Drive or upload files to `/content`, and then runs `thermal_flash_analysis/analyze_flash.py`.

**Before running:** change `REPO_URL` to your GitHub repository URL if needed, and set `SAMPLE_THICKNESS_M` to the measured total package thickness for your module.


In [ ]:
# 1) Clone or update the GitHub repository.
# Replace this with your fork/repository URL if the notebook is copied elsewhere.
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/student-record-storing.git'
REPO_DIR = '/content/student-record-storing'

if 'YOUR_GITHUB_USERNAME' in REPO_URL:
    raise ValueError('Edit REPO_URL first so Colab can clone your GitHub repository.')

import os
import subprocess
from pathlib import Path

if Path(REPO_DIR).exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working in', Path.cwd())


In [ ]:
# 2) Install analysis dependencies.
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'thermal_flash_analysis/requirements.txt'], check=True)


## Video input options

Recommended: put your exported FLIR videos in Google Drive, for example:

```text
MyDrive/FLIR_flash/data/raw/FLIR0626.avi
MyDrive/FLIR_flash/data/raw/FLIR0627.avi
...
```

If you do not want to use Drive, upload videos with the Colab file browser and set `VIDEO_DIR` to that upload folder.


In [ ]:
# 3) Mount Google Drive if your videos are stored there.
# Skip this cell if you uploaded videos directly to /content.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4) Configure your analysis.
from pathlib import Path

VIDEO_DIR = Path('/content/drive/MyDrive/FLIR_flash/data/raw')  # folder containing FLIR0626.avi ... FLIR0650.avi
OUTPUT_DIR = Path('/content/drive/MyDrive/FLIR_flash/results')
MANIFEST = Path('thermal_flash_analysis/experiment_manifest.csv')

SAMPLE_THICKNESS_M = 0.0011  # TODO: replace with your measured total sample/package thickness in metres
ROI = ''  # Optional: 'x,y,width,height', for example '120,80,260,210'. Leave blank for full frame.
MAKE_VIDEO = True  # Writes *_flash_only.avi files for visual inspection.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Video directory:', VIDEO_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
# 5) Run all videos listed in the manifest.
import subprocess
import sys

cmd = [
    sys.executable, 'thermal_flash_analysis/analyze_flash.py',
    '--manifest', str(MANIFEST),
    '--video-dir', str(VIDEO_DIR),
    '--sample-thickness-m', str(SAMPLE_THICKNESS_M),
    '--output-dir', str(OUTPUT_DIR),
]
if ROI:
    cmd.extend(['--roi', ROI])
if MAKE_VIDEO:
    cmd.append('--make-video')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Optional: run one video only instead of the full manifest.
# Uncomment and edit the path when you only want to process one run.
# single_video = VIDEO_DIR / 'FLIR0632.avi'
# cmd = [
#     sys.executable, 'thermal_flash_analysis/analyze_flash.py',
#     '--input', str(single_video),
#     '--sample-thickness-m', str(SAMPLE_THICKNESS_M),
#     '--output-dir', str(OUTPUT_DIR / single_video.stem),
# ]
# if ROI:
#     cmd.extend(['--roi', ROI])
# if MAKE_VIDEO:
#     cmd.append('--make-video')
# subprocess.run(cmd, check=True)


In [ ]:
# 6) View the combined summary table.
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / 'all_summaries.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    display(summary.groupby(['sample', 'configuration'])['effective_diffusivity_m2_s'].agg(['count', 'mean', 'std']))
else:
    print('No all_summaries.csv found yet. Check that videos exist in VIDEO_DIR and match names in the manifest.')


In [ ]:
# 7) Zip results for download if desired.
import shutil
zip_base = '/content/flir_flash_results'
archive = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print('Created', archive)
